<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/logo.png?raw=true" alt="2026年度ゲノム情報解析入門" height="100px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

遺伝子の機能を調べる解析: 配列・ドメイン・立体構造
---

　最後に、遺伝子の機能を調べる方法を紹介していきます。

- BLAST（配列の類似性）
- ドメイン検索（機能をもつ領域）
- タンパク質の性質の計算
- アミノ酸配列の構造予測（AlphaFold）

- (これまでのまとめ)

等を扱います。

# 今回の勉強内容
　今回は、遺伝子の機能を調べる方法をいくつか紹介し、Pythonで実際に体験します。

1. 他の生物で似た配列の遺伝子の機能を調べる
  1. 配列同士のアラインメント
  1. BLAST検索
  1. 自分でアラインメントしてみる
  1. 「似ている」とはどういうことか（スコアとE値）
  1. どこが似ているのか
1. 特定のアミノ酸配列の機能を調べる
  1. ドメイン検索
  1. モチーフを自分で探す
  1. タンパク質の性質を計算する
1. アミノ酸配列の構造予測
  1. AlphaFoldによる予測構造を見る
  1. 実験で決められた構造と比べる
1. これまでのまとめ

---

　一度次のコードセルを実行して、実習に必要なライブラリのインストールと、今回使うアミノ酸配列の準備をしてください。

In [ ]:
#########################
##  実習の前に実行してください。 ##
#########################
!pip install -q biopython py3Dmol

import re
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import Align
from Bio.Align import substitution_matrices

## 今回使うアミノ酸配列
## 機能のわからない遺伝子X（GWASで病気の抵抗性に関わる領域から見つかった、という想定）
gene_X = "MADAVVEFLLLNLKQLLLYHVDLLSGVKDEVESLHRELSLMKAFLKDSREKRSEYEYVRELVSQITIVAYEAEDIIDTFVTNAAMQKARSTVRRAFHVFDHSSKLRNVAKEIESIKVKVKEIYDKKMFGIQSLHGGESSHRSPPQKRVPMVEEENVVGFDDEARKISSRLTNGSEELEIISIVGMGGLGKTTLAKKMEAWDDLQMAFPKTATGSRILLTTRNTEVALHANPEGLPHHLRFLTHEESWELLSKKVFRKGICPLELEDIGLQIAKKCYGLPLAIVVVSGLLLKKEKTRDWWKKIANDVSSYVARDPKQCMDVLALSYKHLPDHLKVCFIYFGVFPEDFEIPVWKLLRLWTSEGFVQKMGQECLEDTAEEYLEDLVDRNLVLVAKKRASGRIKTCRIHDMLRDLSVKMGSEEKFLEVFKESAQNHSLSPISKYHRRLCVHSHFLDFITSRPFGPNVRSFLCFASEEMELLREHTSFLHEAFRLVRVLDLKYINFPRFPTEIVQLVHLRYIALSGNFRVLPASISKLWNLETLIVGTKSRELDIQVDIWKMSQFKHLYTSGLSCLRGPPAKTRKDNADPFVRRNIQTISTVLPDCCKENILARTPGLRKLGIRGKLATLVATNGDSSLFDNLAKLDNLETLKLLNDTFPLPPSQCQIPGLPQSYKFPPNLKKLTLSDTFLDWSHISTLGMLPNLEVLKLKDYAFKGTQWEPLDGGFRLLRVLHIGRTNLEHWNASGHHFPRLQQVFLKHCSSLNEIPFGLVEVSSLQNIELFWPTPAAAASARIIQQEKQEGDIKDNVFKLVIYPPDI"
## シロイヌナズナの ZAR1（病原菌の認識に関わるタンパク質）
ZAR1 = "MVDAVVTVFLEKTLNILEEKGRTVSDYRKQLEDLQSELKYMQSFLKDAERQKRTNETLRTLVADLRELVYEAEDILVDCQLADGDDGNEQRSSNAWLSRLHPARVPLQYKKSKRLQEINERITKIKSQVEPYFEFITPSNVGRDNGTDRWSSPVYDHTQVVGLEGDKRKIKEWLFRSNDSQLLIMAFVGMGGLGKTTIAQEVFNDKEIEHRFERRIWVSVSQTFTEEQIMRSILRNLGDASVGDDIGTLLRKIQQYLLGKRYLIVMDDVWDKNLSWWDKIYQGLPRGQGGSVIVTTRSESVAKRVQARDDKTHRPELLSPDNSWLLFCNVAFAANDGTCERPELEDVGKEIVTKCKGLPLTIKAVGGLLLCKDHVYHEWRRIAEHFQDELRGNTSETDNVMSSLQLSYDELPSHLKSCILTLSLYPEDCVIPKQQLVHGWIGEGFVMWRNGRSATESGEDCFSGLTNRCLIEVVDKTYSGTIITCKIHDMVRDLVIDIAKKDSFSNPEGLNCRHLGISGNFDEKQIKVNHKLRGVVSTTKTGEVNKLNSDLAKKFTDCKYLRVLDISKSIFDAPLSEILDEIASLQHLACLSLSNTHPLIQFPRSMEDLHNLQILDASYCQNLKQLQPCIVLFKKLLVLDMTNCGSLECFPKGIGSLVKLEVLLGFKPARSNNGCKLSEVKNLTNLRKLGLSLTRGDQIEEEELDSLINLSKLMSISINCYDSYGDDLITKIDALTPPHQLHELSLQFYPGKSSPSWLSPHKLPMLRYMSICSGNLVKMQEPFWGNENTHWRIEGLMLSSLSDLDMDWEVLQQSMPYLRTVTANWCPELESFAIEDVGFRGGVWMKTPLHRT"
## イネいもち病菌の AVR-PikD（病原菌がつくるタンパク質）
AVR_PikD = "METGNKYIEKRAIDLSRERDPNFFDHPGIPVPECFWFMFKNNVRQDAGTCYSSWKMDMKVGPNWVHIKSDDNCNLSGDFPPGWIVLGKKRPGF"
## シロイヌナズナの FLS2
FLS2 = "MKLLSKTFLILTLTFFFFGIALAKQSFEPEIEALKSFKNGISNDPLGVLSDWTIIGSLRHCNWTGITCDSTGHVVSVSLLEKQLEGVLSPAIANLTYLQVLDLTSNSFTGKIPAEIGKLTELNQLILYLNYFSGSIPSGIWELKNIFYLDLRNNLLSGDVPEEICKTSSLVLIGFDYNNLTGKIPECLGDLVHLQMFVAAGNHLTGSIPVSIGTLANLTDLDLSGNQLTGKIPRDFGNLLNLQSLVLTENLLEGDIPAEIGNCSSLVQLELYDNQLTGKIPAELGNLVQLQALRIYKNKLTSSIPSSLFRLTQLTHLGLSENHLVGPISEEIGFLESLEVLTLHSNNFTGEFPQSITNLRNLTVLTVGFNNISGELPADLGLLTNLRNLSAHDNLLTGPIPSSISNCTGLKLLDLSHNQMTGEIPRGFGRMNLTFISIGRNHFTGEIPDDIFNCSNLETLSVADNNLTGTLKPLIGKLQKLRILQVSYNSLTGPIPREIGNLKDLNILYLHSNGFTGRIPREMSNLTLLQGLRMYSNDLEGPIPEEMFDMKLLSVLDLSNNKFSGQIPALFSKLESLTYLSLQGNKFNGSIPASLKSLSLLNTFDISDNLLTGTIPGELLASLKNMQLYLNFSNNLLTGTIPKELGKLEMVQEIDLSNNLFSGSIPRSLQACKNVFTLDFSQNNLSGHIPDEVFQGMDMIISLNLSRNSFSGEIPQSFGNMTHLVSLDLSSNNLTGEIPESLANLSTLKHLKLASNNLKGHVPESGVFKNINASDLMGNTDLCGSKKPLKPCTIKQKSSHFSKRTRVILIILGSAAALLLVLLLVLILTCCKKKEKKIENSSESSLPDLDSALKLKRFEPKELEQATDSFNSANIIGSSSLSTVYKGQLEDGTVIAVKVLNLKEFSAESDKWFYTEAKTLSQLKHRNLVKILGFAWESGKTKALVLPFMENGNLEDTIHGSAAPIGSLLEKIDLCVHIASGIDYLHSGYGFPIVHCDLKPANILLDSDRVAHVSDFGTARILGFREDGSTTASTSAFEGTIGYLAPEFAYMRKVTTKADVFSFGIIMMELMTKQRPTSLNDEDSQDMTLRQLVEKSIGNGRKGMVRVLDMELGDSIVSLKQEEAIEDFLKLCLFCTSSRPEDRPDMNEILTHLMKLRGKANSFREDRNEDREV"

print(f"遺伝子X: {len(gene_X)} アミノ酸, ZAR1: {len(ZAR1)} アミノ酸, AVR-PikD: {len(AVR_PikD)} アミノ酸")

# 遺伝子の機能を調べる

これまで学んできた内容で、GWASで候補遺伝子を検出したり、RNA-seqで遺伝子のアノテーションを行ったりと、

「どの配列や領域が重要なのか」ということを調べる手法があったかと思います。

しかし、ほとんどの遺伝子の機能は未だ分かっておらず、候補遺伝子やアノテーションした遺伝子の機能を調べる方法が必要になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/gene_function.png?raw=true" alt="gene_function" height="220px" align="middle">

きっちり機能を調べるには実験的に調べる(着目した遺伝子をノックアウトして形質を見る等)必要がありますが、

今回は解析を元に機能を調べる方法をいくつか紹介したいと思います。

# 1. 他の生物で似た配列の遺伝子の機能を調べる

まず、一番簡単な方法として、他の生物で似たような配列の遺伝子の機能が既に判明していないか調べることです。

特にモデル生物であるシロイヌナズナなどは多くの遺伝子の機能が解明されているので、それらの機能をもとに自分の興味がある生き物の遺伝子の機能を予測を試みることができます。

その時に使用するツールが`BLAST`です。

## BLAST

BLASTとは、Basic Local Alignment Search Toolの頭文字をとったもので、DNAの塩基配列やタンパク質のアミノ酸配列のシーケンスアライメントを行うためのアルゴリズムや、そのアルゴリズムを実装したプログラムの事を指します。

### 配列同士のアラインメント

ここで言うアラインメントとは何なのかと言うと、

塩基配列やアミノ酸配列を、似ている部分を同じ位置になるように並べることを指します。

例えば、次の様な５つのアミノ酸配列があった場合、

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/alignment1.png?raw=true" alt="alignment1" height="80px" align="middle">

これをアラインメントすると以下の様に並べることが出来ます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/alignment2.png?raw=true" alt="alignment2" height="80px" align="middle">

一致している所、不一致な場所、ギャップなどを考慮して、最適なアラインメントを探すための様々なアルゴリズムが開発されています。



### アラインメントを体験する

　Pythonでは、`Biopython` というライブラリでアラインメントをおこなえます。アラインメントでは、次のような点数（スコア）のルールを決め、合計点が最も高くなる並べ方を探します。

- 同じアミノ酸どうし、性質の似たアミノ酸どうしが並ぶと**加点**（どのくらい加点するかは **BLOSUM62** などの置換行列で決める）
- 性質の違うアミノ酸どうしが並ぶと**減点**
- ギャップ（`-`、配列の挿入・欠失）を入れると**減点**（ギャップを開くときの減点と、ギャップを伸ばすときの減点）

　次のコードセルで、2つの短いアミノ酸配列をアラインメントしてみましょう。

`seq2 = "PAWHEAE"`の部分を

とても似ている配列 ex) `seq2 = "HEAGAPGHEE"`や`seq2 = "HEAGAGHEE"`等に変えたり、

全く関係ない配列にして試してみてください。

In [ ]:
seq1 = "HEAGAWGHEE"
seq2 = "PAWHEAE"

aligner = Align.PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")   # 置換行列
aligner.open_gap_score = -8     # ギャップを開くときの減点
aligner.extend_gap_score = -1   # ギャップを1つ伸ばすごとの減点
aligner.mode = "global"         # 配列全体を並べる（global）か、似ている部分だけを並べる（local）か

alignment = aligner.align(seq1, seq2)[0]
print(f"スコア: {alignment.score}")
print(alignment)

　アラインメントの結果は、ギャップに対する減点をどこまで許容するか等、点数のルールによって変わります。「唯一の正しいアラインメント」があるわけではない、という点に注意してください。

### BLAST検索

BLASTの話に戻りますが、一般的にBLASTと言うと、データベースの中から、入力配列に似た配列を検索してくれるツールの事を指します。

最も有名なものはNCBIの[BLAST検索](https://blast.ncbi.nlm.nih.gov/Blast.cgi)です。御存知の方も多いのではないでしょうか。

入力した配列に似た配列をNCBIに登録されているデータベースから検索してくれるツールになります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/BLAST.png?raw=true" alt="BLAST" height="280px" align="middle">

例えば今回、GWASの結果、病気の抵抗性に重要だと考えられる領域が見つかり、その領域には機能が分かっていない遺伝子があったとします。

その遺伝子のコードするアミノ酸配列が、 `MADAVVEFLLLNLKQLLLYHVDLLSGVKDEVESLHRELSLMKAFLKDSREKRSEYEYVRELVSQITIVAYEAEDIIDTFVTNAAMQKARSTVRRAFHVFDHSSKLRNVAKEIESIKVKVKEIYDKKMFGIQSLHGGESSHRSPPQKRVPMVEEENVVGFDDEARKISSRLTNGSEELEIISIVGMGGLGKTTLAKKMEAWDDLQMAFPKTATGSRILLTTRNTEVALHANPEGLPHHLRFLTHEESWELLSKKVFRKGICPLELEDIGLQIAKKCYGLPLAIVVVSGLLLKKEKTRDWWKKIANDVSSYVARDPKQCMDVLALSYKHLPDHLKVCFIYFGVFPEDFEIPVWKLLRLWTSEGFVQKMGQECLEDTAEEYLEDLVDRNLVLVAKKRASGRIKTCRIHDMLRDLSVKMGSEEKFLEVFKESAQNHSLSPISKYHRRLCVHSHFLDFITSRPFGPNVRSFLCFASEEMELLREHTSFLHEAFRLVRVLDLKYINFPRFPTEIVQLVHLRYIALSGNFRVLPASISKLWNLETLIVGTKSRELDIQVDIWKMSQFKHLYTSGLSCLRGPPAKTRKDNADPFVRRNIQTISTVLPDCCKENILARTPGLRKLGIRGKLATLVATNGDSSLFDNLAKLDNLETLKLLNDTFPLPPSQCQIPGLPQSYKFPPNLKKLTLSDTFLDWSHISTLGMLPNLEVLKLKDYAFKGTQWEPLDGGFRLLRVLHIGRTNLEHWNASGHHFPRLQQVFLKHCSSLNEIPFGLVEVSSLQNIELFWPTPAAAASARIIQQEKQEGDIKDNVFKLVIYPPDI`

という配列だったとしましょう。

<br>

この時にまず[BLAST検索](https://blast.ncbi.nlm.nih.gov/Blast.cgi)をかけてみて、似たような配列でかつ機能が分かっている遺伝子が無いか探してみる、という方法を取ります。

実際に検索してみましょう。

今回はアミノ酸配列を検索したいので`Protein BLAST(blastp)`を使用します。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/blast1.png?raw=true" alt="ZAR1_blast" height="250px" align="middle">

検索したい配列を一番上の枠に入力して、一番下にあるBLASTボタンを押すとBLASTが開始されます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/blast2.png?raw=true" alt="ZAR1_blast" height="450px" align="middle">

<br>

この結果から、似た配列を持つ遺伝子がNCBIに登録されていれば、その機能を元に着目している遺伝子の機能を推定することが出来ます。

ただし、"似た配列"の遺伝子の機能が分かるだけであり、配列が少し違うだけで大きく機能が異なる遺伝子もたくさんあるので、あくまでも参考程度な情報になります。

<br>

#### 保存性の検索

また、BLASTは着目した遺伝子がどのくらい広い種に保存されているかを調べるのにも用いられたりします。

例えば以下の配列はシロイヌナズナのZAR1という遺伝子の配列です。

ZAR1:

`MVDAVVTVFLEKTLNILEEKGRTVSDYRKQLEDLQSELKYMQSFLKDAERQKRTNETLRTLVADLRELVYEAEDILVDCQLADGDDGNEQRSSNAWLSRLHPARVPLQYKKSKRLQEINERITKIKSQVEPYFEFITPSNVGRDNGTDRWSSPVYDHTQVVGLEGDKRKIKEWLFRSNDSQLLIMAFVGMGGLGKTTIAQEVFNDKEIEHRFERRIWVSVSQTFTEEQIMRSILRNLGDASVGDDIGTLLRKIQQYLLGKRYLIVMDDVWDKNLSWWDKIYQGLPRGQGGSVIVTTRSESVAKRVQARDDKTHRPELLSPDNSWLLFCNVAFAANDGTCERPELEDVGKEIVTKCKGLPLTIKAVGGLLLCKDHVYHEWRRIAEHFQDELRGNTSETDNVMSSLQLSYDELPSHLKSCILTLSLYPEDCVIPKQQLVHGWIGEGFVMWRNGRSATESGEDCFSGLTNRCLIEVVDKTYSGTIITCKIHDMVRDLVIDIAKKDSFSNPEGLNCRHLGISGNFDEKQIKVNHKLRGVVSTTKTGEVNKLNSDLAKKFTDCKYLRVLDISKSIFDAPLSEILDEIASLQHLACLSLSNTHPLIQFPRSMEDLHNLQILDASYCQNLKQLQPCIVLFKKLLVLDMTNCGSLECFPKGIGSLVKLEVLLGFKPARSNNGCKLSEVKNLTNLRKLGLSLTRGDQIEEEELDSLINLSKLMSISINCYDSYGDDLITKIDALTPPHQLHELSLQFYPGKSSPSWLSPHKLPMLRYMSICSGNLVKMQEPFWGNENTHWRIEGLMLSSLSDLDMDWEVLQQSMPYLRTVTANWCPELESFAIEDVGFRGGVWMKTPLHRT`

これをBLAST検索することでZAR1に似た遺伝子がどのくらいたくさんの植物種に存在しているかを調べることが出来ます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/ZAR1_blast.png?raw=true" alt="ZAR1_blast" height="350px" align="middle">

#### BLASTの定義(細かい話ですが...)

最初に説明した通り、BLASTは配列のシーケンスアライメントを行うためのアルゴリズムや、そのアルゴリズムを実装したプログラムの事を指します。

「BLASTする」と言うと「NCBIのBLAST検索をする」を表すと思っている人も多いのですが、BLASTはNCBIのサービスのことではなく、そういうプログラム全体の事を指します。

よって「イネで見つけた遺伝子Xの配列をコムギのリファレンスゲノム配列に対してBLASTする」みたいな時にも使える言葉になります。



## アラインメントの一致率についてもう少し見てみる

　BLAST検索では、データベースの中の数億本の配列の中から似た配列を探すために、計算を速くする工夫がされています。また、結果には30%程度しかアミノ酸が一致していない様な配列も結果として出てきます。このくらいの一致ではあまり意味が無さそうに見えますがどうでしょうか？

　上で紹介した、機能のわからない遺伝子X と、シロイヌナズナの ZAR1 を比べて確認してみます。

下のコードで遺伝子XとZAR1の配列をアラインメントしてみます。

In [ ]:
aligner = Align.PairwiseAligner(mode="local", substitution_matrix=substitution_matrices.load("BLOSUM62"),
                                open_gap_score=-10, extend_gap_score=-0.5)   # BLASTの標準に近い設定

def compare(a, b, show=True):
    """2つの配列をアラインメントし、スコアと一致率（identity）を表示する"""
    aln = aligner.align(a, b)[0]
    c = aln.counts()
    identity = c.identities / (c.identities + c.mismatches) * 100
    if show:
        print(f"スコア: {aln.score:.0f}")
        print(f"同じアミノ酸: {c.identities}, 違うアミノ酸: {c.mismatches}, ギャップ: {c.gaps}")
        print(f"一致率（identity）: {identity:.1f}%")
    return aln

aln_X_ZAR1 = compare(gene_X, ZAR1)
print()
print(str(aln_X_ZAR1)[:1000])   # アラインメントの最初の部分を表示

　一致率は約33%でした。3分の1しか同じでないので、あまり似ていないように感じるかもしれません。しかし、タンパク質の場合、一致率が30%程度あれば、立体構造や機能が似ていることが多いと言われています。アラインメントの `|` は同じアミノ酸、`.` は違うアミノ酸です。

## 「似ている」とはどういうことか（スコアとE値）

　では、スコア578や一致率は約33%は「似ている」と言える値なのでしょうか？ これを判断するために、**偶然でもどのくらいのスコアが出るのか**を調べてみましょう。ZAR1のアミノ酸の並びをランダムに並べ替えた配列（アミノ酸の種類と数は同じで、並び方だけでたらめな配列）を作り、遺伝子Xとのスコアを計算します。

In [ ]:
random.seed(0)
real_score = aligner.score(gene_X, ZAR1)

shuffled_scores = []
for i in range(200):
    shuffled = list(ZAR1)
    random.shuffle(shuffled)                     # ZAR1のアミノ酸の並びをでたらめにする
    shuffled_scores.append(aligner.score(gene_X, "".join(shuffled)))

print(f"本物のZAR1とのスコア          : {real_score:.0f}")
print(f"並べ替えた配列とのスコア（200回）: 平均 {np.mean(shuffled_scores):.0f}, 最大 {np.max(shuffled_scores):.0f}")

plt.hist(shuffled_scores, bins=20, label="shuffled ZAR1")
plt.axvline(real_score, color="red", label="real score")
plt.xlabel("Alignment score"); plt.ylabel("Count"); plt.legend()
plt.show()

　並べ替えた配列とのスコアは、200回やっても最大で100程度でした。本物のZAR1とのスコア（578）は、偶然ではまず出ない値なので、遺伝子XとZAR1は**進化的に共通の祖先に由来する**（**相同な**）配列だと考えられます。

　BLAST検索の結果に表示される **E値（E-value）** も、同じ考え方の数値です。E値は「同じ大きさのデータベースを、でたらめな配列で検索したときに、そのスコア以上のヒットが偶然いくつ見つかると期待されるか」を表します。E値が小さいほど（例: 1e-50）、偶然では説明できない、意味のある類似だと考えられます。


　また、どんな2つの配列でも、アラインメントすれば何かしらの並べ方とスコアが得られます。ローカルアラインメントは、似ている部分だけを取り出すので、短い区間がたまたま似ているだけでも一致率が高く見えることがあります。**一致率だけでなく、スコアやE値、並んだ区間の長さ**を合わせて判断することが大切です。

## どこが似ているのか

　一言で「配列が似ている遺伝子」と言っても、全体的にまんべんなく似ているのか、それとも部分的によく似ているのか、様々です。

　遺伝子XとZAR1は、配列全体で一様に似ているのでしょうか？ ZAR1の配列に沿って、50アミノ酸ごとの一致率を計算してみましょう。図の縦線は、後の「2. 特定のアミノ酸配列の機能を調べる」で探す、機能的に重要なモチーフの位置です。

In [ ]:
aln = aligner.align(ZAR1, gene_X)[0]
same = np.zeros(len(ZAR1))                 # ZAR1の各位置で、遺伝子Xと同じアミノ酸なら1
covered = np.zeros(len(ZAR1))              # 遺伝子Xのアミノ酸と並んだ位置なら1
for (s1, e1), (s2, e2) in zip(*aln.aligned):
    for k in range(e1 - s1):
        covered[s1 + k] = 1
        same[s1 + k] = ZAR1[s1 + k] == gene_X[s2 + k]

window = 50
identity = [same[i:i + window].mean() * 100 for i in range(len(ZAR1) - window)]
plt.figure(figsize=(12, 3))
plt.plot(range(window // 2, len(ZAR1) - window // 2), identity)
plt.fill_between(range(len(ZAR1)), 0, 100, where=covered == 0, color="gray", alpha=0.3, label="gap in gene X")
for name, pos in [("P-loop", 188), ("GLPL", 356), ("MHD", 486)]:
    plt.axvline(pos, color="red", ls="--", lw=1)
    plt.text(pos + 5, 85, name, color="red")
plt.xlabel("Position in ZAR1 (amino acid)"); plt.ylabel(f"Identity in {window} aa window (%)")
plt.ylim(0, 100); plt.legend(loc="upper right")
plt.show()

#### 確認ポイント
- 一致率が比較的高いのは、ZAR1のどのあたりでしょうか。赤い縦線のモチーフ（NB-ARCドメインの中にあるモチーフ）は機能的に重要な配列の部分です。
- 灰色の部分は、遺伝子Xに対応する配列がない（ギャップになった）場所です。遺伝子Xでは、ZAR1の約200〜270番目にあたる部分がまるごと欠けていました。この部分に機能的に重要な配列があったとしたら、遺伝子Xの機能はZAR1と同じだと言えるかどうか怪しくなります（「2. 特定のアミノ酸配列の機能を調べる」で確かめます）

#### (発展) PythonからBLAST検索をする
　BiopythonからNCBIのBLAST検索を実行することもできます。次のコードセルは、遺伝子Xを、機能が詳しく調べられているタンパク質のデータベース（Swiss-Prot）に対してBLAST検索します。混雑状況によって数分以上かかったり、失敗したりすることがあるので、時間に余裕があるときに試してください。

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML

try:
    handle = NCBIWWW.qblast("blastp", "swissprot", gene_X, hitlist_size=10, expect=1e-10)
    record = NCBIXML.read(handle)
    rows = []
    for hit in record.alignments:
        hsp = hit.hsps[0]
        rows.append({"ヒットした配列": hit.title[:90], "E値": hsp.expect,
                     "一致率(%)": round(hsp.identities / hsp.align_length * 100, 1)})
    display(pd.DataFrame(rows))
except Exception as e:
    print("BLAST検索に失敗しました。時間をおいて再度試してください:", e)

# 2. 特定のアミノ酸配列の機能を調べる

## ドメイン検索

それでは次に、BLAST検索で良い結果が得られなかった場合や、一致率が高い配列の機能を予測した場合など、違ったアプローチで機能を予測したいときに用いられるドメイン検索について見ていきます。

タンパク質は、立体構造をとりますが、その中に構造的にまとまった独立して安定した三次構造の一部の領域(ドメイン)を持つことが多いです。

そのため、調べたいアミノ酸配列に対し、そのような機能を持つ領域(ドメイン)をデータベースから検索し、機能予測に役立てることが出来ます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/domains.png?raw=true" alt="InterPro" height="280px" align="middle">

<br><br>

最も良く使用されるドメイン検索ツールとしては、[InterProScan](http://www.ebi.ac.uk/interpro/search/sequence/)があります。

InterProScanは複数のデータベースを元にドメイン検索を行ってくれます。

先ほどのZAR1遺伝子の配列を使って検索してみましょう。

検索には少し時間がかかります。


##### ZAR1の配列

```
MVDAVVTVFLEKTLNILEEKGRTVSDYRKQLEDLQSELKYMQSFLKDAERQKRTNETLRTLVADLRELVYEAEDILVDCQLADGDDGNEQRSSNAWLSRLHPARVPLQYKKSKRLQEINERITKIKSQVEPYFEFITPSNVGRDNGTDRWSSPVYDHTQVVGLEGDKRKIKEWLFRSNDSQLLIMAFVGMGGLGKTTIAQEVFNDKEIEHRFERRIWVSVSQTFTEEQIMRSILRNLGDASVGDDIGTLLRKIQQYLLGKRYLIVMDDVWDKNLSWWDKIYQGLPRGQGGSVIVTTRSESVAKRVQARDDKTHRPELLSPDNSWLLFCNVAFAANDGTCERPELEDVGKEIVTKCKGLPLTIKAVGGLLLCKDHVYHEWRRIAEHFQDELRGNTSETDNVMSSLQLSYDELPSHLKSCILTLSLYPEDCVIPKQQLVHGWIGEGFVMWRNGRSATESGEDCFSGLTNRCLIEVVDKTYSGTIITCKIHDMVRDLVIDIAKKDSFSNPEGLNCRHLGISGNFDEKQIKVNHKLRGVVSTTKTGEVNKLNSDLAKKFTDCKYLRVLDISKSIFDAPLSEILDEIASLQHLACLSLSNTHPLIQFPRSMEDLHNLQILDASYCQNLKQLQPCIVLFKKLLVLDMTNCGSLECFPKGIGSLVKLEVLLGFKPARSNNGCKLSEVKNLTNLRKLGLSLTRGDQIEEEELDSLINLSKLMSISINCYDSYGDDLITKIDALTPPHQLHELSLQFYPGKSSPSWLSPHKLPMLRYMSICSGNLVKMQEPFWGNENTHWRIEGLMLSSLSDLDMDWEVLQQSMPYLRTVTANWCPELESFAIEDVGFRGGVWMKTPLHRT
```


<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/InterPro.png?raw=true" alt="InterPro" height="280px" align="middle">

こうして得られたドメイン情報を元に、配列の機能予測を行ったりします。

<br>

#### アラインメントとドメイン情報を組み合わせる

似たような配列なのに機能が異なる配列があった場合には、先ほど紹介したアラインメントとドメイン情報を組み合わせて情報を得る場合もあります。

アラインメントは全配列を並べる形ですが、配列が一致しなかったりギャップがあった時に、その場所が機能的に重要かどうか、という情報があるとより機能的な違いの解釈がしやすくなる可能性があります。

単純にアラインメントして違いがどれだけあるか、という情報に加えて、ドメインとして認識された場所にどれだけ違いがあるのか、を調べる形です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/domain_alignment.png?raw=true" alt="domain_alignment" height="120px" align="middle">

## モチーフを自分で探す

　InterProScanなどのドメイン検索は、ドメインやモチーフを表す「パターン」や「プロファイル」をデータベースにまとめておき、入力した配列と照らし合わせることで動いています。最も単純な形式が、[PROSITE](https://prosite.expasy.org/) というデータベースの**パターン**です。

　たとえば、ATPやGTPなどのヌクレオチドに結合するタンパク質の多くがもつ **P-loop**（Walker Aモチーフ）は、PROSITEでは次のパターンで表されます（PS00017）。

```
[AG]-x(4)-G-K-[ST]
```

　これは「AかG、その後に任意のアミノ酸が4つ、G、K、SかT」という意味です。下記コードではPythonでこのドメインを検索できるようにしてあります。

　これで先ほどの遺伝子X、ZAR1に加えて、AVR-PikDというイネいもち病菌がつくるタンパク質に対して調べてみます。

```
AVR_PikD: METGNKYIEKRAIDLSRERDPNFFDHPGIPVPECFWFMFKNNVRQDAGTCYSSWKMDMKVGPNWVHIKSDDNCNLSGDFPPGWIVLGKKRPGF
```

In [ ]:
def prosite_to_regex(pattern):
    """PROSITEのパターンを、Pythonの正規表現に書き直す"""
    regex = pattern.replace("-", "").replace("x", ".").replace("(", "{").replace(")", "}")
    regex = regex.replace("<", "^").replace(">", "$")
    return re.sub(r"\{([A-Z]+)\}", r"[^\1]", regex)   # {P} は「P以外」

def find_motif(sequence, regex):
    """配列の中でモチーフが見つかった位置（1から数える）と、見つかった配列を返す"""
    return [(m.start() + 1, m.group(1)) for m in re.finditer(f"(?=({regex}))", sequence)]

p_loop = prosite_to_regex("[AG]-x(4)-G-K-[ST]")
print("正規表現:", p_loop)
for name, seq in [("遺伝子X", gene_X), ("ZAR1", ZAR1), ("AVR-PikD", AVR_PikD)]:
    print(f"{name}: {find_motif(seq, p_loop)}")

　遺伝子XとZAR1には、どちらにもP-loopが1つずつ見つかりました。AVR-PikDには見つかりませんでした。

### NLRタンパク質のモチーフ

　ZAR1は、植物が病原菌を認識する受容体である **NLR**（Nucleotide-binding Leucine-rich Repeat）タンパク質の1つです。NLRタンパク質の中央にある **NB-ARC ドメイン**（ヌクレオチドに結合して、受容体のスイッチとして働く部分）には、P-loopのほかにも、よく保存されたモチーフがいくつかあります。

In [ ]:
## NB-ARCドメインの代表的なモチーフ（正規表現は、この実習用に簡略化したもの）
nlr_motifs = {
    "P-loop (Walker A)": "[AG].{4}GK[ST]",
    "Walker B":          "[LIVMF]{4}D[DE]",
    "GLPL":              "GLPL",
    "MHD":               "[IVLM]HD[ML]",
}
table = pd.DataFrame({name: {motif: ", ".join(str(p) for p, _ in find_motif(seq, regex)) or "なし"
                             for motif, regex in nlr_motifs.items()}
                      for name, seq in [("遺伝子X", gene_X), ("ZAR1", ZAR1), ("AVR-PikD", AVR_PikD)]})
table   # 数字はモチーフが見つかった位置（何番目のアミノ酸か）

#### 確認ポイント
- 遺伝子Xには、ZAR1と同じ順番で P-loop、GLPL、MHD が並んでいました。遺伝子Xも、ZAR1と同じNLRタンパク質の仲間である可能性が高そうです。
- 一方、遺伝子Xには **Walker B** モチーフが見つかりませんでした。Walker Bは、ZAR1では263番目付近、つまり「どこが似ているのか」の図で、遺伝子Xに対応する配列がなかった（灰色の）部分にあります。Walker Bは、ATPの分解に関わる重要なモチーフです。このモチーフが欠けている遺伝子Xは、ZAR1と同じように働くのかはわかりません。

　このように、**アラインメント（どこが似ているか）とドメイン・モチーフの情報（どこが機能的に重要か）を組み合わせる**ことで、配列の違いが機能に影響しそうかを考える手がかりが得られます。

<small>※ ここで使ったPythonのコードは、授業用に簡略化したものです。実際のドメイン検索では、InterProScanのように、たくさんの配列から作られた統計的なモデル（隠れマルコフモデルなど）を使って、より正確にドメインを検出します。</small>

## タンパク質の性質を計算する

　アミノ酸配列からは、分子量や等電点（タンパク質の電荷が0になるpH）など、タンパク質の基本的な性質も計算できます。

In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

rows = []
for name, seq in [("遺伝子X", gene_X), ("ZAR1", ZAR1), ("AVR-PikD", AVR_PikD)]:
    p = ProteinAnalysis(seq)
    rows.append({"タンパク質": name, "長さ": len(seq),
                 "分子量 (kDa)": round(p.molecular_weight() / 1000, 1),
                 "等電点": round(p.isoelectric_point(), 2),
                 "疎水性 (GRAVY)": round(p.gravy(), 3),
                 "ロイシンの割合 (%)": round(seq.count("L") / len(seq) * 100, 1)})
pd.DataFrame(rows)

#### 確認ポイント

- **ロイシンの割合**を比べてみましょう。NLRタンパク質のC末端側には、ロイシンに富んだ繰り返し配列（**LRR**、Leucine-Rich Repeat）があります。遺伝子XとZAR1のロイシンの割合がAVR-PikDと比べて高くなっている原因です。
- 次のコードセルで、配列に沿った疎水性（Kyte-Doolittleの指標、9アミノ酸ごとの平均）を描いてみましょう。値が大きいほど疎水性（水になじみにくい）です。長く続く疎水性の領域は、細胞膜を貫通する部分であることがあります。

今回は`gene-X`と`ZAR1`に加えて、膜貫通タンパクである`FLS2`を比較します。

`seq` を`gene_X, ZAR1, FLS2`のいずれかに変えて、3つのタンパク質の疎水性領域を比べてみてください。

In [ ]:
seq = ZAR1      # gene_X, ZAR1, FLS2 から選ぶ

kd = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
      "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6, "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
window = 9
hydro = [np.mean([kd[a] for a in seq[i:i + window]]) for i in range(len(seq) - window + 1)]
plt.figure(figsize=(12, 3))
plt.plot(range(window // 2 + 1, len(seq) - window // 2 + 1), hydro)
plt.axhline(0, color="black", lw=0.5)
plt.xlabel("Position (amino acid)"); plt.ylabel("Hydropathy (Kyte-Doolittle)")
plt.show()

FLS2では短いですが800番目付近に疎水性の領域があるかと思います。

# 3. アミノ酸配列の構造予測

最後に、タンパク質の構造予測です。アミノ酸配列は、立体構造を取ることで機能を持つことになるため、

あるアミノ酸配列がどんな立体構造になるのかという情報は機能を知る意味でも重要です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/protein_structure.png?raw=true" alt="protein_structure" height="200px" align="middle">

実験的にタンパク質の構造を確認する手法としては、X線や電子顕微鏡等を用いて結晶構造を解析する方法がありますが、

特定の遺伝子(アミノ酸配列)を結晶化することが難しく、難易度の高い実験になります。

そこで、実験的な手法に対して、バイオインフォマティクスでは機械学習等の技術を用いてタンパク質の立体構造を予測する研究が行われていました。

立体構造の予測は、数年前に公開されたAlphaFold2と呼ばれるモデルが非常に高い精度を出したことで有名になり、

現在では更に進化したAlphaFold3というモデルが一般の人でも使える様に整備されています。

[AlphaFold3](https://alphafoldserver.com/) から使うことが出来ます。

例えば

AVR-PikDという遺伝子の立体構造を調べてみましょう。
`METGNKYIEKRAIDLSRERDPNFFDHPGIPVPECFWFMFKNNVRQDAGTCYSSWKMDMKVGPNWVHIKSDDNCNLSGDFPPGWIVLGKKRPGF`

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/alphafold3.png?raw=true" alt="similar_structure" height="300px" align="middle">

アミノ酸配列を入力し、ジョブを投げることで、そのアミノ酸配列がとり得る構造を予測してくれます。

複数のアミノ酸配列をまとめて入れることで、構造的に結合するかどうかを予測する事も出来ます。

(生物先端コースの学生実験では、構造による結合予測や構造解析ソフトなども詳しくやる予定です。)

<br>

ある2つの遺伝子が配列として大きく違っていても立体構造にすると似ている場合、機能としても似た機能を持っていた、というケースも報告されているので、

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/similar_structure.png?raw=true" alt="similar_structure" height="220px" align="middle">

この様なアプローチを元に、立体構造を予測し、配列レベルではなく構造レベルで似ている遺伝子を探す、という方法が使えたりします。

しかし、AlphaFold3は精度が高いとはいえ、100%正しいわけではなく、構造が変化するタンパク質の予測は苦手だったり、まだまだ問題は残っています。

また、予測はあくまでも予測に過ぎないので、これを結論とするのではなく、参考程度の情報として捉える位が良いかと思います。

## AlphaFoldによる予測構造を見る

　AlphaFoldで予測された構造は、[AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/)（AlphaFold DB）に、2億以上のタンパク質についてまとめて公開されています。UniProtのID（タンパク質ごとの番号）で検索できます。ZAR1のUniProt IDは **Q38834** です。

　次のコードセルで、AlphaFold DBからZAR1の予測構造をダウンロードし、`py3Dmol` というライブラリで表示します。マウスで回転・拡大できます。

　色は、AlphaFoldが予測の確からしさを表す **pLDDT**（0〜100）です。
- 青: pLDDT が高い（90以上ならとても確か）
- 赤: pLDDT が低い（50未満の部分は、構造をとらない柔らかい領域であることも多い）

In [ ]:
import py3Dmol

uniprot_id = "Q38834"   # ZAR1

## AlphaFold DB から予測構造（PDB形式）をダウンロードする
info = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}").json()[0]
af_pdb = requests.get(info["pdbUrl"]).text
print(f"{info.get('uniprotDescription', '')}（{info.get('organismScientificName', '')}）")

## 表示（pLDDT で色分け: 赤 → 黄 → 緑 → 青 の順に確か）
view = py3Dmol.view(width=700, height=500)
view.addModel(af_pdb, "pdb")
view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
view.zoomTo()
view.show()

　pLDDTを、配列に沿ったグラフにもしてみましょう。予測構造のファイルでは、各原子の「B-factor」という欄にpLDDTが書かれています。

In [ ]:
## 各アミノ酸のα炭素（CA）の行から、位置とpLDDTを取り出す
plddt = [(int(line[22:26]), float(line[60:66])) for line in af_pdb.splitlines()
         if line.startswith("ATOM") and line[12:16].strip() == "CA"]
pos, score = zip(*plddt)

plt.figure(figsize=(12, 3))
plt.plot(pos, score)
for name, p in [("P-loop", 188), ("GLPL", 356), ("MHD", 486)]:
    plt.axvline(p, color="red", ls="--", lw=1); plt.text(p + 5, 30, name, color="red")
plt.axhline(70, color="gray", ls=":", lw=1)
plt.xlabel("Position in ZAR1 (amino acid)"); plt.ylabel("pLDDT"); plt.ylim(0, 100)
plt.show()
print(f"pLDDTの平均: {np.mean(score):.1f}")

### 練習3-1
- pLDDTが低い（予測が不確かな）のは、ZAR1のどのあたりでしたか？
- 上のコードセルの `uniprot_id` を、他のタンパク質のIDに変えてみましょう。UniProt（https://www.uniprot.org/ ）で好きなタンパク質を検索すると、IDがわかります

  - 例: イネのいもち病抵抗性に関わる転写因子 OsWRKY45
  - 例: 緑色蛍光タンパク質 GFP
  - 例: ヘモグロビン Hemoglobin
  - 等々

## 実験で決められた構造と比べる

　ZAR1は、病原菌のタンパク質を認識すると、5つのZAR1分子が集まって車輪のような形の複合体（**レジストソーム**、resistosome）を作ることが、クライオ電子顕微鏡による構造解析で明らかになりました（Wang et al. 2019, *Science* 364: eaav5870）。この複合体は細胞膜に穴をあけるように働き、細胞死を引き起こして病原菌の増殖を抑えると考えられています。

　実験で決められた構造は、[Protein Data Bank（PDB）](https://www.rcsb.org/) に登録されています。ZAR1のレジストソームの構造（PDB ID: **6J5T**）を表示してみましょう。

In [ ]:
pdb_id = "6J5T"
pdb_text = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb").text

## 鎖（チェーン）ごとのアミノ酸の数を数える
chains = {}
for line in pdb_text.splitlines():
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        chains[line[21]] = chains.get(line[21], 0) + 1
print(f"チェーンの数: {len(chains)}")
print(chains)

## 表示（ZAR1: オレンジ、RKS1: 緑、PBL2: 紫）
groups = {"ZAR1": (["C", "F", "G", "L", "O"], "orange"),
          "RKS1": (["B", "E", "H", "K", "N"], "green"),
          "PBL2": (["A", "D", "I", "J", "M"], "purple")}
view = py3Dmol.view(width=700, height=500)
view.addModel(pdb_text, "pdb")
for name, (chain_ids, color) in groups.items():
    view.setStyle({"chain": chain_ids}, {"cartoon": {"color": color}})
view.zoomTo()
view.show()

#### 確認ポイント
- 複合体は、ZAR1（オレンジ）、RKS1（緑）、PBL2（紫）の3種類のタンパク質が5つずつ、合計15本の鎖でできています。上から見ると、5つのZAR1が輪のように並んでいることがわかります。
- AlphaFold DBの予測構造は、ZAR1が1分子だけの構造でした。複合体の形は、1分子の予測からはわかりません。AlphaFold3（AlphaFold Server）では、複数の配列を入力して、複合体の構造を予測することもできます。

<small>※ Google Colab上でAlphaFold2を動かせる [ColabFold](https://github.com/sokrypton/ColabFold)（Mirdita et al. 2022, *Nature Methods* 19: 679–682）というツールもあります。GPUを使うので、Colabの設定やタンパク質の長さによっては時間がかかります。</small>

# 遺伝子の機能を調べる方法のまとめ

紹介してきた様に、バイオインフォマティクスでは、

着目した遺伝子の配列の類似性や、データベースに対して検索をかけることで、様々な情報を得ることが出来ます。

機能が未知の遺伝子に関してはこの様な情報を元に機能を予測する形になります。

ただし、あくまでも機能に関係していそうな情報が手に入るだけであり、最終的には実験的に確認しないと正しい機能は分からない、という点は頭に入れておく必要があります。

# DNA-seq, RNA-seq等を用いた解析のまとめ

　ここまで、DNA-seqやRNA-seq等のシーケンス情報をどの様に扱うのか、それらを用いてどの様な解析を行うことが出来るのかを主に学んできました。

　DNA配列をシーケンスすることで、ある生物のゲノム情報を読み取ることが出来、また、交配集団等のシーケンス情報からSNP等の変異を検出し、GWASやQTL-seq等を通して、QTLを検出することが出来ます。

　RNA-seqを行うことで、遺伝子の発現量を定量化することが可能になり、着目した現象に関係する遺伝子を発現量という観点から調べることが出来ます。また、遺伝子ネットワークの様な遺伝子間の関係性を調べる際にもRNA-seqは有用なアプローチの1つです。

　これらの解析技術は次世代シーケンサー技術の発展・コスト減に伴ってますます広がりを見せ、生物学の課題を解決するための重要なアプローチとなっています。

　そのため、将来的にデータ解析的な研究をする人は当然習得していくべき技術ですが、そうでない人にとっても、課題を解決する際に、ゲノム解析やバイオインフォマティクスによってどの様なことが出来るのか、何をしているのか、といった大まかな知識を持っておけると選択肢として役立つことがあるかもしれません。

　また、皆さんが研究をする際に、どの様な専門分野に進んだとしても、ゲノム解析やバイオインフォマティクスによるアプローチで課題解決を試みる研究というのは存在しており、そういった研究を理解する為にも知識を持っておけると良いのではないかなと思います。

### この授業で学んだことの振り返り

|分野|主な回の内容|わかること|
|:--|:--|:--|
|ゲノムを読む|ゲノムアセンブル|生物のゲノム配列そのもの|
|違いを見つける|リードアライメント・SNP検出|個体や品種の間のDNAの違い|
|形質の原因を探す|QTL-seq、GWAS|形質に関わる遺伝子の場所|
|形質を予測する|Genomic Prediction|ゲノムの情報から形質値を予測し、選抜に使う|
|発現を調べる|RNA-seq（発現解析・ネットワーク・eQTL）|どの遺伝子が、いつ、どこで、どのくらい働くか|
|機能を推定する|BLAST・ドメイン検索・構造予測|見つけた遺伝子が、どんな働きをもちそうか|

　たとえば、「病気に強い品種をつくりたい」という目標があったとします。GWASやQTL-seqで抵抗性に関わる領域を見つけ、RNA-seqで病原菌に応答して発現が変わる遺伝子を絞り込み、今回の方法でその遺伝子の機能を推定する。そして、見つけた遺伝子の情報をGenomic Predictionやマーカー選抜に活かす。このように、これまでの解析はつながっています。

> ### 課題
> **この授業で学んだ解析の中で、自分の研究や興味のあるテーマに一番使えそうだと思ったものは何ですか？ その理由も書いてください。**
>
> <small>自由記述で回答してください。</small>